## Clase 7 - Ejercicio: algoritmo de búsqueda para la Torre de Hanoi

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [8]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [9]:
def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    discos = [i for i in range(number_disks, 0, -1)]
    estado_inicial = StatesHanoi(discos, [], [], max_disks=number_disks)
    estado_objetivo = StatesHanoi([], [], discos, max_disks=number_disks)
    problema = ProblemHanoi(initial=estado_inicial, goal=estado_objetivo)

    ##### EDITAR ESTA ZONA

    # Uso A* para elegir el nodo con menor costo recorrido + heurística.
    from heapq import heappop, heappush
    from itertools import count

    def heuristica(estado):
        # Cuento los discos que ya están bien ubicados desde la base.
        varilla_destino = estado.get_state()[2]
        discos_bien_ubicados = 0

        for posicion, disco in enumerate(varilla_destino):
            if disco != number_disks - posicion:
                break
            discos_bien_ubicados += 1

        # Los de afuera deben llegar; los mal ubicados deben salir y volver.
        discos_fuera = number_disks - len(varilla_destino)
        discos_mal_ubicados = len(varilla_destino) - discos_bien_ubicados
        return float(discos_fuera + 2 * discos_mal_ubicados)

    raiz = NodeHanoi(estado_inicial)
    # El contador evita comparar nodos cuando tienen la misma prioridad.
    desempate = count()
    frontera = []
    h_inicial = heuristica(raiz.state)
    heappush(frontera, (raiz.path_cost + h_inicial, h_inicial, next(desempate), raiz))

    # Guardo el mejor costo encontrado para cada estado.
    mejor_costo = {estado_inicial: raiz.path_cost}
    costos_en_frontera = {estado_inicial: raiz.path_cost}
    estados_visitados = {estado_inicial}
    estados_explorados = set()

    solucion = None
    nodos_explorados = 0
    profundidad_maxima = 0

    while frontera:
        _, _, _, nodo = heappop(frontera)

        # Salteo los estados ya explorados o que tienen un camino mejor.
        if nodo.state in estados_explorados:
            continue
        if nodo.path_cost != mejor_costo.get(nodo.state):
            continue

        costos_en_frontera.pop(nodo.state, None)
        estados_explorados.add(nodo.state)
        nodos_explorados += 1
        profundidad_maxima = max(profundidad_maxima, nodo.depth)

        if problema.goal_test(nodo.state):
            solucion = nodo
            break

        for hijo in nodo.expand(problema):
            estados_visitados.add(hijo.state)
            costo_anterior = mejor_costo.get(hijo.state, float("inf"))

            if hijo.path_cost < costo_anterior:
                mejor_costo[hijo.state] = hijo.path_cost
                costos_en_frontera[hijo.state] = hijo.path_cost

                # Si encuentro un camino mejor, vuelvo a considerar el estado.
                estados_explorados.discard(hijo.state)
                h = heuristica(hijo.state)
                f = hijo.path_cost + h
                heappush(
                    frontera,
                    (f, h, next(desempate), hijo),
                )

    # Mantengo los nombres de las métricas que pide la consigna.
    metricas = {
        "solution_found": solucion is not None,
        "nodes_explored": nodos_explorados,
        "states_visited": len(estados_visitados),
        "nodes_in_frontier": len(costos_en_frontera),
        "max_depth": profundidad_maxima,
        "cost_total": float(solucion.path_cost) if solucion else 0.0,
    }
    #####

    return solucion, metricas

Se prueba la función:

In [10]:
solucion, metricas = search_algorithm(number_disks=5)

Veamos las métricas:

In [11]:
for nombre, valor in metricas.items():
    print(f"{nombre}: {valor}")

solution_found: True
nodes_explored: 148
states_visited: 159
nodes_in_frontier: 11
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [12]:
for nodo in solucion.path():
    print(nodo)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [13]:
for accion in solucion.solution():
    print(accion)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
